# Numerical Integration
This section runs several numerical integration methods on common probability density functions. It demonstrates how grid-based rules (rectangle, trapezoid, Simpson) behave on finite and truncated infinite intervals, and prepares the data for interpreting the printed outputs.


## Section 1 — Density Setup and Uniform-Grid Integration
This section defines a small suite of probability density functions (Uniform, Normal, Exponential, Laplace, Cauchy, Triangular) and then evaluates their integrals using uniform-grid rules (rectangle, trapezoid, Simpson). Because several PDFs are defined on infinite domains, the code explicitly **truncates** them to finite windows (e.g., Normal on [-6,6]).

**Key points to keep in mind:**
- The uniform-grid rules are deterministic and use evenly spaced nodes.
- For finite-support PDFs (Uniform, Triangular), the integral should be close to 1.0 on the exact support.
- For infinite-support PDFs, results depend on both grid density and the chosen truncation window; widening the window reduces truncation error but may require more points.
- Simpson’s rule should generally outperform rectangle/trapezoid for smooth PDFs at the same grid density.


In [31]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("/Users/I768838/Desktop/Probabilistic-quadrature")))

import numpy as np
import matplotlib.pyplot as plt

from source.functions.function import Function
from source.functions.domain import Domain
from source.functions.interval import Interval
from source.numeric_integration.numeric_integral_factory.numeric_integral_factory import NumericIntegralFactory
from source.numeric_integration.numeric_integration_pattern import NumericIntegrationPattern


factory = NumericIntegralFactory()


def construct_some_probability_density_function() -> list[Function]:
    a, b, c, mu, sigma, lam, b_lap, gamma = 0, 1, 0.5, 0, 1, 1, 1, 1

    uniform_density_function = Function(
        lambda x: 1 / (b - a), 
        Domain(Interval(a, b)), 
        None, "Uniform(a,b)"
    )

    normal_density_function = Function(
        lambda x: (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-((x - mu) ** 2) / (2 * sigma ** 2)), 
        Domain(Interval(float("-inf"), float("inf"))), 
        None, "Normal(mu,sigma)"
    )

    exp_density = Function(
        lambda x: lam * np.exp(-lam * x), 
        Domain(Interval(0.0, float("inf"))), 
        None, "Exp(lambda)"
    )

    laplace_density = Function(
        lambda x: (1 / (2 * b_lap)) * np.exp(-np.abs(x - mu) / b_lap), 
        Domain(Interval(float("-inf"), float("inf"))), 
        None, "Laplace(mu,b)")

    cauchy_density = Function(
        lambda x: (1 / (np.pi * gamma * (1 + ((x - mu) / gamma) ** 2))), 
        Domain(Interval(float("-inf"), float("inf"))), 
        None, "Cauchy(mu,gamma)")

    tri_density = Function(
        lambda x: (2*(x-a)/((b-a)*(c-a)) if a <= x < c else 2*(b-x)/((b-a)*(b-c)) if c <= x <= b else 0.0), 
        Domain(Interval(a, b)), 
        None, "Triangular(a,b,c)")

    return [
        uniform_density_function, normal_density_function,
        exp_density, laplace_density, cauchy_density, tri_density
    ]


def finite_interval_for_density(function: Function) -> Interval:
    if function.name.startswith("Normal"): return Interval(-6, 6)
    if function.name.startswith("Laplace"): return Interval(-10, 10)
    if function.name.startswith("Cauchy"): return Interval(-25, 25)
    if function.name.startswith("Exp"): return Interval(0.0, 10)
    return function.domain.get_intervals()


def uniform_grid_integration(function: Function, interval: Interval, point_density: int) -> None:
    x_coordinates = list(np.linspace(interval.get_left_component(), interval.get_right_component(), point_density))

    rectangle_integration_method = factory.create(NumericIntegrationPattern.RECTANGLE, None, function, interval, x_coordinates, point_density - 1)
    simpson_integration_method = factory.create(NumericIntegrationPattern.SIMPSON, None, function, interval, x_coordinates, point_density - 1)
    trapezoid_integration_method = factory.create(NumericIntegrationPattern.TRAPEZOID, None, function, interval, x_coordinates, point_density - 1)

    print(f"\n{function.name} on [{interval.get_left_component()}, {interval.get_right_component()}]")
    print(f"Rectangle : {rectangle_integration_method.integrate():.8f}")
    print(f"Simpson   : {simpson_integration_method.integrate():.8f}")
    print(f"Trapezoid : {trapezoid_integration_method.integrate():.8f}\n")
    

probability_density_functions = construct_some_probability_density_function()
for current_probability_density_function in probability_density_functions:
    interval = finite_interval_for_density(current_probability_density_function)
    uniform_grid_integration(current_probability_density_function, interval, 1000)


Uniform(a,b) on [0.0, 1.0]
Rectangle : 1.00000000
Simpson   : 1.00000000
Trapezoid : 1.00000000


Normal(mu,sigma) on [-6.0, 6.0]
Rectangle : 1.00000000
Simpson   : 1.00000000
Trapezoid : 1.00000000


Exp(lambda) on [0.0, 10.0]
Rectangle : 0.99995043
Simpson   : 0.99995460
Trapezoid : 0.99996295


Laplace(mu,b) on [-10.0, 10.0]
Rectangle : 0.99998800
Simpson   : 0.99997130
Trapezoid : 0.99993790


Cauchy(mu,gamma) on [-25.0, 25.0]
Rectangle : 0.97454879
Simpson   : 0.97454878
Trapezoid : 0.97454876


Triangular(a,b,c) on [0.0, 1.0]
Rectangle : 1.00000100
Simpson   : 1.00000033
Trapezoid : 0.99999900



## Uniform Grid Methods — Output Interpretation
This section prints three approximations for each density on a finite interval:
- `Rectangle`: midpoint rule on a uniform grid.
- `Simpson`: Simpson’s rule on the same grid (higher-order accuracy for smooth functions).
- `Trapezoid`: trapezoidal rule on the same grid.

**What to expect in the output:**
- For `Uniform(a,b)` and `Triangular(a,b,c)` the interval is exactly `[a,b]`, so the integral should be very close to `1.0`.
- For `Normal`, `Laplace`, `Cauchy`, and `Exp` the domain is infinite, so we integrate on a **finite window** (e.g., `[-6,6]` or `[0,10]`). The values should be **close to 1**, but can be slightly below/above depending on tail mass outside the window and the grid resolution.
- Increasing `point_density` tightens the grid and usually improves accuracy, but does not fix truncation error from finite windows. To reduce truncation error, widen the interval.


## Section 2 — Weighted-Node (Gaussian) Quadrature
This section applies Gauss-type quadratures. These methods are optimized for specific weights and domains:
- **Legendre** for finite intervals without extra weights.
- **Hermite** for (-∞,∞) with weight e^{-x^2}.
- **Laguerre** for [0,∞) with weight e^{-x}.
- **Chebyshev** for [-1,1] with weight 1/sqrt(1-x^2).

**Interpretation notes:**
- The reported values are **weighted integrals** when Hermite/Laguerre/Chebyshev are used; they are not the plain integral of the PDF.
- For PDFs on finite intervals, Gauss–Legendre should produce values close to 1.0 with enough nodes.
- The number of nodes (n) controls accuracy; Gauss rules often converge faster than uniform-grid rules for smooth integrands.


In [32]:
def weighted_nodes_integration(function: Function, count_of_nodes: int) -> None:
    interval = function.domain.get_intervals()
    if isinstance(interval, list):
        interval = interval[0]

    left = interval.get_left_component()
    right = interval.get_right_component()
    print(f"\n{function.name} (weighted nodes, n={count_of_nodes})")

    if np.isfinite(left) and np.isfinite(right):
        leg = factory.create(NumericIntegrationPattern.LEGENDRE, None, function, count_of_nodes, interval)
        print(f"Gauss-Legendre : {leg.integrate():.8f}")

        if np.isclose(left, -1.0) and np.isclose(right, 1.0):
            cheb = factory.create(NumericIntegrationPattern.CHEBYSHEV, None, function, count_of_nodes)
            print(f"Gauss-Chebyshev: {cheb.integrate():.8f}")
        else:
            print("Gauss-Chebyshev: skipped (requires interval [-1, 1])")

    if np.isneginf(left) and np.isposinf(right):
        herm = factory.create(NumericIntegrationPattern.HERMITE, None, function, count_of_nodes)
        print(f"Gauss-Hermite  : {herm.integrate():.8f}")

    if np.isclose(left, 0.0) and np.isposinf(right):
        lag = factory.create(NumericIntegrationPattern.LAGUERRE, None, function, count_of_nodes)
        print(f"Gauss-Laguerre : {lag.integrate():.8f}")


for current_probability_density_function in probability_density_functions:
    weighted_nodes_integration(current_probability_density_function, 100)


Uniform(a,b) (weighted nodes, n=100)
Gauss-Legendre : 1.00000000
Gauss-Chebyshev: skipped (requires interval [-1, 1])

Normal(mu,sigma) (weighted nodes, n=100)
Gauss-Hermite  : 0.57735027

Exp(lambda) (weighted nodes, n=100)
Gauss-Laguerre : 0.50000000

Laplace(mu,b) (weighted nodes, n=100)
Gauss-Hermite  : 0.54358070

Cauchy(mu,gamma) (weighted nodes, n=100)
Gauss-Hermite  : 0.42758358

Triangular(a,b,c) (weighted nodes, n=100)
Gauss-Legendre : 0.99991856
Gauss-Chebyshev: skipped (requires interval [-1, 1])


## Weighted-Node Methods — Output Interpretation
This section runs Gauss quadrature rules that are **defined with specific weight functions**:
- `Gauss-Legendre` integrates on a **finite interval** without extra weights.
- `Gauss-Hermite` integrates on `(-∞, ∞)` with weight `e^{-x^2}`.
- `Gauss-Laguerre` integrates on `[0, ∞)` with weight `e^{-x}`.
- `Gauss-Chebyshev` integrates on `[-1, 1]` with weight `1/sqrt(1-x^2)`.

**How to read the numbers:**
- For finite-domain PDFs (`Uniform`, `Triangular`) the Legendre result should be very close to `1.0`.
- For infinite-domain PDFs, the Hermite/Laguerre values **are not the same as the plain integral** of the PDF, because of the weight. These outputs demonstrate *weighted integration*, not “area = 1”.
- Chebyshev appears only when the interval is exactly `[-1, 1]` in this implementation; otherwise it is skipped.

**Controlling accuracy:**
- Increase `count_of_nodes` for higher accuracy in Gauss rules.
- Choose functions that match the rule’s weight (e.g., Hermite for Gaussian-like integrands, Laguerre for exponential tails) to see best performance.


## Section 3 — Monte Carlo Integration
This section demonstrates Monte Carlo integration on finite intervals. The method samples from a uniform measure on the chosen interval and estimates the integral via sample averages. It includes:
- **Standard MC** (plain sampling).
- **Weighted MC** (importance-style but with constant weights in this demo).
- **Recursive MC** (subdivides the interval and estimates recursively).


In [37]:
from source.numeric_integration.monte_carlo.monte_carlo_factory.monte_carlo_factory import MonteCarloFactory
from source.measures.uniform_box_measure import UniformBoxMeasure
from source.numeric_integration.monte_carlo.monte_carlo_stretegies import MonteCarloIntegrationStrategy
from source.random_variables.continuous_random_variables.uniform import Uniform
from source.functions.measured_function import MeasuredFunction


monte_carlo_method_factory = MonteCarloFactory()


def eval_function_on_samples(X):
    X = np.atleast_2d(X)
    return np.array([function(x) for x in X[:, 0]])


def monte_carlo_integration(function: Function, interval: Interval, n_samples: int = 20000, depth: int = 4) -> None:
    a = interval.get_left_component()
    b = interval.get_right_component()
    if not (np.isfinite(a) and np.isfinite(b)):
        raise ValueError("Monte Carlo demo needs a finite interval.")

    measure = UniformBoxMeasure(np.array([a]), np.array([b]))
    random_variable = Uniform(a, b)
    mc_function = MeasuredFunction(eval_function_on_samples, measure, input_name=function.name)

    mc_standard = monte_carlo_method_factory.create(
        MonteCarloIntegrationStrategy.STANDARD,
        mc_function, measure, n_samples=n_samples, rv=random_variable
    )
    mc_weighted = monte_carlo_method_factory.create(
        MonteCarloIntegrationStrategy.WEIGHTED,
        mc_function, measure, n_samples=n_samples, rv=random_variable,
        proposal_sampler=lambda n, r: r.sample(n),
        weight_fn=lambda x: np.ones(x.shape[0])
    )
    mc_recursive = monte_carlo_method_factory.create(
        MonteCarloIntegrationStrategy.RECURSIVE,
        mc_function, measure, n_samples=n_samples, rv=random_variable, depth=depth
    )

    volume = b - a
    std_est = mc_standard.integrate() * volume
    wgt_est = mc_weighted.integrate() * volume
    rec_est = mc_recursive.integrate()

    std_err = mc_standard.stderr * volume if mc_standard.stderr else None
    wgt_err = mc_weighted.stderr * volume if mc_weighted.stderr else None

    print(f"\n{function.name} MC on [{a}, {b}] (n={n_samples})")
    print(f"MC standard : {std_est:.8f}" + (f"  stderr≈{std_err:.2e}" if std_err else ""))
    print(f"MC weighted : {wgt_est:.8f}" + (f"  stderr≈{wgt_err:.2e}" if wgt_err else ""))
    print(f"MC recursive: {rec_est:.8f}")


for current_probability_density_function in probability_density_functions:
    interval = finite_interval_for_density(current_probability_density_function)
    monte_carlo_integration(current_probability_density_function, interval, n_samples=20000, depth=5)


Uniform(a,b) MC on [0.0, 1.0] (n=20000)
MC standard : 0.33210881  stderr≈2.12e-03
MC weighted : 0.33728252  stderr≈2.11e-03
MC recursive: 0.33322305

Normal(mu,sigma) MC on [-6.0, 6.0] (n=20000)
MC standard : 143.77248989  stderr≈9.08e-01
MC weighted : 143.93998839  stderr≈9.08e-01
MC recursive: 144.04324896

Exp(lambda) MC on [0.0, 10.0] (n=20000)
MC standard : 329.86793468  stderr≈2.10e+00
MC weighted : 336.36840141  stderr≈2.12e+00
MC recursive: 333.36479094

Laplace(mu,b) MC on [-10.0, 10.0] (n=20000)
MC standard : 667.70323039  stderr≈4.21e+00
MC weighted : 665.94871739  stderr≈4.21e+00
MC recursive: 666.88064803

Cauchy(mu,gamma) MC on [-25.0, 25.0] (n=20000)
MC standard : 10395.85754035  stderr≈6.62e+01
MC weighted : 10398.20308947  stderr≈6.58e+01
MC recursive: 10422.02424141

Triangular(a,b,c) MC on [0.0, 1.0] (n=20000)
MC standard : 0.33168608  stderr≈2.09e-03
MC weighted : 0.33437494  stderr≈2.11e-03
MC recursive: 0.33335338


## Monte Carlo — Output Interpretation
- `MC standard` is the plain estimator (mean of samples, scaled by interval length).
- `MC weighted` uses the same samples and constant weights here, so it should be close to `MC standard`.
- `MC recursive` reports the recursive subdivision estimate; it can differ slightly because it allocates samples per subinterval.
- `stderr` is an estimated standard error (only for standard/weighted); values shrink as `n_samples` grows.

**How to read the output:**
- MC estimates are stochastic: expect small deviations around the true value with standard error ~ O(1/sqrt(N)).
- For truncated infinite-domain PDFs, the estimate reflects both sampling noise and truncation error.
- Increasing `n_samples` reduces variance but does not fix truncation bias; for that you must widen the interval.


## Section 4 — Bayesian Quadrature
This section demonstrates Bayesian Quadrature (BQ) on a 1D integrand under a Gaussian measure. The method fits a Gaussian Process (GP) to function evaluations and then computes a posterior distribution for the integral.


In [49]:
from source.kernels.rbf_kernel import RBFKernel
from source.kernels.matern32_kernel import Matern32Kernel
from source.measures.gaussian_measure import GaussianMeasure
from source.measures.measure import Measure


def construct_some_test_functions() -> list[Function]:
    domain = Domain(Interval(float("-inf"), float("inf")))
    return [
        Function(lambda x: x**2, domain, None, "x^2"),
        Function(lambda x: np.sin(x), domain, None, "sin(x)"),
        Function(lambda x: np.cos(x), domain, None, "cos(x)"),
        Function(lambda x: np.exp(-x**2), domain, None, "exp(-x^2)"),
        Function(lambda x: x**4, domain, None, "x^4"),
        Function(lambda x: np.tanh(x), domain, None, "tanh(x)"),
        Function(lambda x: np.sin(2*x) + 0.5*np.cos(3*x), domain, None, "sin(2x)+0.5cos(3x)")
    ]


def bayesian_quadrature_integration(function: Function, measure: Measure, count_of_nodes: int = 20, want_rbf: bool = True,
    lengthscale: float = 1.0, variance: float = 1.0, noise: float = 0.0, jitter: float = 1e-8) -> None:
    X = measure.sample(count_of_nodes)
    y = np.array([function(x[0]) for x in X])
    if want_rbf:
        kernel = RBFKernel(lengthscale=lengthscale, variance=variance)
    else:
        kernel = Matern32Kernel(lengthscale=lengthscale, variance=variance)

    bayesian_quadratue = factory.create(
        NumericIntegrationPattern.BAYESIAN, None,
        function, measure, kernel, X, y, noise=noise, jitter=jitter
    )

    estimate = bayesian_quadratue.integrate()
    variance_post = bayesian_quadratue.variance
    print(f"\n{function.name} (BQ, n={count_of_nodes})")
    print(f"Estimate: {estimate:.6f}")
    print(f"Posterior variance: {variance_post:.6e}")


measure = GaussianMeasure(mean=np.array([0.0]), cov=np.array([[1.0]]))
functions = construct_some_test_functions()

for current_test_function in functions:
    bayesian_quadrature_integration(current_test_function, measure, count_of_nodes=20, want_rbf=True)
    bayesian_quadrature_integration(current_test_function, measure, count_of_nodes=20, want_rbf=False)



x^2 (BQ, n=20)
Estimate: 0.943557
Posterior variance: 3.420781e-05

x^2 (BQ, n=20)
Estimate: 0.891035
Posterior variance: 0.000000e+00

sin(x) (BQ, n=20)
Estimate: 0.000887
Posterior variance: 1.002193e-04

sin(x) (BQ, n=20)
Estimate: -0.004192
Posterior variance: 0.000000e+00

cos(x) (BQ, n=20)
Estimate: 0.611113
Posterior variance: 4.625399e-05

cos(x) (BQ, n=20)
Estimate: 0.605760
Posterior variance: 9.510709e-03

exp(-x^2) (BQ, n=20)
Estimate: 0.578097
Posterior variance: 1.385118e-05

exp(-x^2) (BQ, n=20)
Estimate: 0.575233
Posterior variance: 6.556309e-03

x^4 (BQ, n=20)
Estimate: 2.476091
Posterior variance: 2.662756e-05

x^4 (BQ, n=20)
Estimate: 1.809816
Posterior variance: 0.000000e+00

tanh(x) (BQ, n=20)
Estimate: -0.000061
Posterior variance: 2.400610e-06

tanh(x) (BQ, n=20)
Estimate: 0.015894
Posterior variance: 0.000000e+00

sin(2x)+0.5cos(3x) (BQ, n=20)
Estimate: 0.011327
Posterior variance: 2.164794e-05

sin(2x)+0.5cos(3x) (BQ, n=20)
Estimate: 0.014615
Posterior varianc

## Bayesian Quadrature — Output Interpretation
- The **estimate** is the posterior mean of the integral under the GP model.
- The **variance** is the model’s uncertainty about the integral; it should decrease as you add more observations.
- The true value here is 1.0 because E[x^2] under N(0,1) equals the variance of the normal distribution.

**Notes:**
- Results depend on kernel choice, lengthscale, and the number/placement of sample points.
- The kernel mean/variance terms are approximated by Monte Carlo inside the model, so small fluctuations are expected.
- Increasing the number of training points or using active acquisition typically reduces posterior variance.
